In [ ]:
import json
from pathlib import Path

import torch
import torch.nn as nn
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

from adam_type import RMSpropAdaptiveMomentum
from sgd_am import SGDAdaptiveMomentum
from fsgd_am import FractionalSGDAdaptiveMomentum


DATASET_NAME = "cifar100"
NUM_CLASSES = 100

NUM_EPOCH = 500
BATCH_SIZE = 256
NUM_WORKERS = 8

OPTIMIZER_NAME = "fsgd_am"
MODEL_NAME = "resnet50"
OUTPUT_DIR = Path("outputs")


TRAIN_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

VAL_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])


def setup_dataloader(dataset_name: str):
    if dataset_name == "cifar10":
        train_dataset = datasets.CIFAR10(
            root="./data", train=True, download=True, transform=TRAIN_TRANSFORM
        )
        val_dataset = datasets.CIFAR10(
            root="./data", train=False, download=True, transform=VAL_TRANSFORM
        )

    elif dataset_name == "cifar100":
        train_dataset = datasets.CIFAR100(
            root="./data", train=True, download=True, transform=TRAIN_TRANSFORM
        )
        val_dataset = datasets.CIFAR100(
            root="./data", train=False, download=True, transform=VAL_TRANSFORM
        )
    else:
        raise ValueError(f"{dataset_name} not supported")

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    return train_loader, val_loader


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    train_loader, val_loader = setup_dataloader(DATASET_NAME)

    model = models.resnet50(num_classes=NUM_CLASSES)
    model = model.to(device)

    # =============================
    # Optimizer
    # =============================
    if OPTIMIZER_NAME == "adam_type":
        optimizer = RMSpropAdaptiveMomentum(
            model.parameters(),
            lr=0.1,
            alpha=0.99,
            eps=1,
            weight_decay=5e-7,
            momentum=0.9
        )

    elif OPTIMIZER_NAME == "sgd_am":
        optimizer = SGDAdaptiveMomentum(
            model.parameters(),
            lr=0.1,
            momentum=0.9,
            weight_decay=5e-7
        )

    elif OPTIMIZER_NAME == "fsgd_am":
        optimizer = FractionalSGDAdaptiveMomentum(model.parameters())
        print("Using fsgd_am Optimizer")

    else:
        raise ValueError("optimizer not supported")

    scheduler = CosineAnnealingLR(
        optimizer,
        T_max=NUM_EPOCH,
        eta_min=1e-4
    )

    criterion = nn.CrossEntropyLoss()

    # =============================
    # History lưu metric
    # =============================
    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
        "lr": []
    }

    # =============================
    # Train loop
    # =============================
    for epoch in range(NUM_EPOCH):

        # -------------------------
        # TRAIN
        # -------------------------
        model.train()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for batch_idx, (data, target) in enumerate(train_loader):

            data = data.to(device)
            target = target.to(device)

            optimizer.zero_grad()

            output = model(data)

            loss = criterion(output, target)

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * data.size(0)

            pred = output.argmax(dim=1)
            train_correct += (pred == target).sum().item()
            train_total += target.size(0)

            if batch_idx % 100 == 0:
                print(
                    f"[Train] Epoch {epoch} "
                    f"Batch {batch_idx}/{len(train_loader)} "
                    f"Loss {loss.item():.4f}"
                )

        avg_train_loss = train_loss / train_total
        train_acc = 100.0 * train_correct / train_total

        # -------------------------
        # VALIDATION
        # -------------------------
        model.eval()

        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for batch_idx, (data, target) in enumerate(val_loader):

                data = data.to(device)
                target = target.to(device)

                output = model(data)
                loss = criterion(output, target)

                val_loss += loss.item() * data.size(0)

                pred = output.argmax(dim=1)
                val_correct += (pred == target).sum().item()
                val_total += target.size(0)

        avg_val_loss = val_loss / val_total
        val_acc = 100.0 * val_correct / val_total

        # scheduler sau epoch
        scheduler.step()

        current_lr = optimizer.param_groups[0]["lr"]

        # -------------------------
        # Save history
        # -------------------------
        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["lr"].append(current_lr)

        print(
            f"Epoch [{epoch+1}/{NUM_EPOCH}] | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Train Acc: {train_acc:.2f}% | "
            f"Val Loss: {avg_val_loss:.4f} | "
            f"Val Acc: {val_acc:.2f}% | "
            f"LR: {current_lr:.6f}"
        )

    # =============================
    # Save history json
    # =============================
    save_path = OUTPUT_DIR / f"{DATASET_NAME}_{MODEL_NAME}_{OPTIMIZER_NAME}_history.json"

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=4)

    print(f"Saved history: {save_path}")


if __name__ == "__main__":
    main()

Using fsgd_am Optimizer
[Train] Epoch 0 Batch 0/196 Loss 4.7514
[Train] Epoch 0 Batch 100/196 Loss 4.5942
Epoch [1/500] | Train Loss: 4.6120 | Train Acc: 1.46% | Val Loss: 4.5605 | Val Acc: 2.23% | LR: 0.001000
[Train] Epoch 1 Batch 0/196 Loss 4.5615
[Train] Epoch 1 Batch 100/196 Loss 4.5235
Epoch [2/500] | Train Loss: 4.5292 | Train Acc: 2.08% | Val Loss: 4.5002 | Val Acc: 2.40% | LR: 0.001000
[Train] Epoch 2 Batch 0/196 Loss 4.5096
[Train] Epoch 2 Batch 100/196 Loss 4.4612
Epoch [3/500] | Train Loss: 4.4867 | Train Acc: 2.69% | Val Loss: 4.4809 | Val Acc: 2.32% | LR: 0.001000
[Train] Epoch 3 Batch 0/196 Loss 4.5023
[Train] Epoch 3 Batch 100/196 Loss 4.4356
Epoch [4/500] | Train Loss: 4.4631 | Train Acc: 3.06% | Val Loss: 4.4649 | Val Acc: 2.63% | LR: 0.001000
[Train] Epoch 4 Batch 0/196 Loss 4.4062
